# Demonstration 5: sub-Poissonian occupancy

**What this validates:** the *second* moment of the occupancy distribution.

Demonstrations 1 to 4 all check a first moment: a density profile, an equilibrium
ratio, a chemical potential. Those can all be right while the fluctuations are wrong.
This one measures the variance, which is an independent consequence of the same free
energy.

**The two distributions.** An ideal lattice gas puts each particle in a voxel
independently, so single-voxel occupancy is Poisson and

$$\frac{\mathrm{Var}(n)}{\langle n\rangle} = 1.$$

Hard spheres cannot do that. A voxel that already holds particles is expensive to add
to, so the occupancy distribution narrows. It becomes **sub-Poissonian**.

**The analytic prediction.** The compressibility relation ties the variance to the
chemical potential. With $\beta\mu = \ln\rho + \beta\mu_{\rm ex}(\rho)$,

$$\frac{\mathrm{Var}(n)}{\langle n\rangle}
  = \frac{1}{\rho\,\partial(\beta\mu)/\partial\rho}
  = \frac{1}{1 + \eta\,\dfrac{d(\beta\mu_{\rm ex})}{d\eta}}.$$

For an ideal gas $\mu_{\rm ex} = 0$ and the ratio is exactly 1. For hard spheres
$\mu_{\rm ex}' > 0$, so the ratio falls below 1. Carnahan-Starling supplies
$\mu_{\rm ex}(\eta)$, so the prediction is parameter-free.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species, mu_ex_carnahan_starling
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import report_comparison, relative_discrepancy

## Two finite-size corrections, both small

Be explicit about these, because the claim is a variance and variances are where
finite-size effects hide.

1. **The lattice is canonical, not grand-canonical.** Total particle number is fixed,
   so the variance of the *total* is zero. The relation above is grand-canonical. It
   applies to one voxel because one voxel out of $V$ is a small subvolume, and the
   rest of the lattice acts as its reservoir. The correction is $(1 - 1/V)$.
2. **The ideal control is multinomial, not exactly Poisson.** Placing $N$ particles
   into $V$ voxels gives each voxel variance $\langle n\rangle(1 - 1/V)$. So the
   ideal case is also short of 1 by the same factor.

With $V = 4096$ both corrections are $0.02\%$, far below the sampling error. The
predicted ideal value is therefore taken as $1 - 1/V$, not 1.

In [ ]:
SHAPE      = (64, 64)
VOXEL_NM   = 20.0
SIGMA_NM   = 8.0            # hard-sphere diameter for the excluded case
CAP        = 20
OCCUPANCIES = (3, 6)        # particles per voxel -> eta = 0.10 and 0.20
N_STEPS    = 40_000
BURN_IN    = N_STEPS // 3
SAMPLE_EVERY = 100          # occupancy decorrelates slowly; space the samples

V     = SHAPE[0] * SHAPE[1]
dxi3  = (np.pi / 6) * SIGMA_NM ** 3 / VOXEL_NM ** 3
FINITE_SIZE = 1.0 - 1.0 / V

def predicted_ratio(eta, h=1e-6):
    """1 / (1 + eta * d(mu_ex)/d eta), times the finite-size factor."""
    if eta == 0.0:
        return FINITE_SIZE
    dmu = (mu_ex_carnahan_starling(eta + h)
           - mu_ex_carnahan_starling(eta - h)) / (2 * h)
    return FINITE_SIZE / (1.0 + eta * dmu)

print(f"one particle contributes dxi3 = {dxi3:.5f}")
print(f"finite-size factor (1 - 1/V)  = {FINITE_SIZE:.5f}")
for n in OCCUPANCIES:
    print(f"  {n} per voxel -> eta = {n * dxi3:.3f}, "
          f"predicted Var/mean = {predicted_ratio(n * dxi3):.4f}")

## Measure

No field, so the equilibrium state is uniform and every voxel is equivalent. That
lets us pool the occupancy over all 4096 voxels at each sample, which is what makes a
variance measurable in reasonable time.

The ideal case sets `sigma_nm = 0` and `exclusion=False`. It is a true control: the
same transport code, the same sampling, only the free energy removed.

In [ ]:
def measure(n_per_voxel, excluded):
    """Return (mean, variance, n_samples) of single-voxel occupancy at equilibrium."""
    eta = n_per_voxel * dxi3 if excluded else 0.0
    tau = (suggest_tau(1.0, VOXEL_NM, 2, max(3.0 * eta, 0.05)) if excluded else 2e-5)
    sim = Simulation(
        shape=SHAPE, voxel_nm=VOXEL_NM,
        species=[Species("A", SIGMA_NM if excluded else 0.0, np.zeros(1))],
        occupancy_cap=CAP if excluded else 10_000,
        psi=np.zeros((1,) + SHAPE), D_um2_s=1.0, tau_s=tau,
        exclusion=excluded, seed=1, attach_log_handler=False,
    )
    sim.set_counts("A", np.full(SHAPE, n_per_voxel, dtype=np.int64))
    sim.record_initial()

    s = s2 = 0.0
    n = 0
    for i in range(N_STEPS):
        sim.step()
        if i >= BURN_IN and i % SAMPLE_EVERY == 0:
            c = sim.state.counts[0].astype(np.float64)
            s += c.sum()
            s2 += (c * c).sum()
            n += c.size
    sim.state.check_mass()
    mean = s / n
    return mean, s2 / n - mean * mean, n // V, tau

results = []
for n_pv in OCCUPANCIES:
    for excluded in (False, True):
        mean, var, snaps, tau = measure(n_pv, excluded)
        results.append(dict(n_pv=n_pv, excluded=excluded,
                            eta=n_pv * dxi3 if excluded else 0.0,
                            mean=mean, var=var, ratio=var / mean, snaps=snaps))
print(f"{len(results)} runs complete")

In [ ]:
print(f"{'case':>10} {'n/voxel':>8} {'eta':>7} {'mean':>7} {'var':>7} "
      f"{'Var/mean':>9} {'predicted':>10} {'error':>7}")
for r in results:
    pred = predicted_ratio(r["eta"])
    err = abs(r["ratio"] / pred - 1)
    print(f"{'excluded' if r['excluded'] else 'ideal':>10} {r['n_pv']:>8} "
          f"{r['eta']:>7.3f} {r['mean']:>7.3f} {r['var']:>7.3f} "
          f"{r['ratio']:>9.4f} {pred:>10.4f} {err*100:>6.1f}%")

print()
ideal = [r for r in results if not r["excluded"]]
excl  = [r for r in results if r["excluded"]]
print(report_comparison(
    "ideal lattice gas: Var/mean vs Poisson",
    float(np.mean([r["ratio"] for r in ideal])), FINITE_SIZE))
print()
for r in excl:
    print(report_comparison(
        f"excluded, eta = {r['eta']:.3f}: Var/mean vs compressibility relation",
        r["ratio"], predicted_ratio(r["eta"])))

VERDICTS = {
    "ideal is Poisson": max(abs(r["ratio"] / FINITE_SIZE - 1) for r in ideal),
    "excluded matches compressibility":
        max(abs(r["ratio"] / predicted_ratio(r["eta"]) - 1) for r in excl),
}

In [ ]:
eta_curve = np.linspace(0.0, 0.30, 200)
pred_curve = np.array([predicted_ratio(e) for e in eta_curve])

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))

ax[0].plot(eta_curve, pred_curve, "-", lw=1.8, color="#c1440e",
           label="compressibility relation", zorder=2)
ax[0].axhline(FINITE_SIZE, ls="--", lw=1.2, color="#888",
              label="Poisson (ideal gas)")
ax[0].plot([r["eta"] for r in excl], [r["ratio"] for r in excl], "o", ms=7,
           color="#1f4e79", label="measured, exclusion on", zorder=3)
ax[0].plot([0.0] * len(ideal), [r["ratio"] for r in ideal], "s", ms=7,
           color="#2e7d32", label="measured, ideal", zorder=3)
ax[0].set_xlabel(r"packing fraction $\eta = \xi_3$")
ax[0].set_ylabel(r"$\mathrm{Var}(n)\,/\,\langle n\rangle$")
ax[0].set_title("Exclusion narrows the occupancy distribution", fontsize=10)
ax[0].set_ylim(0, 1.15)
ax[0].legend(frameon=False, fontsize=8.5)

# the distributions themselves, at the denser setting
n_pv = OCCUPANCIES[-1]
for excluded, colour, label in ((False, "#2e7d32", "ideal"),
                                (True, "#1f4e79", "exclusion on")):
    eta = n_pv * dxi3 if excluded else 0.0
    tau = (suggest_tau(1.0, VOXEL_NM, 2, max(3.0 * eta, 0.05)) if excluded else 2e-5)
    sim = Simulation(
        shape=SHAPE, voxel_nm=VOXEL_NM,
        species=[Species("A", SIGMA_NM if excluded else 0.0, np.zeros(1))],
        occupancy_cap=CAP if excluded else 10_000,
        psi=np.zeros((1,) + SHAPE), D_um2_s=1.0, tau_s=tau,
        exclusion=excluded, seed=2, attach_log_handler=False,
    )
    sim.set_counts("A", np.full(SHAPE, n_pv, dtype=np.int64))
    sim.record_initial()
    for i in range(N_STEPS // 2):
        sim.step()
    c = sim.state.counts[0]
    bins = np.arange(-0.5, c.max() + 1.5)
    ax[1].hist(c, bins=bins, density=True, histtype="step", lw=1.8,
               color=colour, label=f"{label}  (var/mean = {c.var()/c.mean():.2f})")
ax[1].set_xlabel("occupancy of a single voxel")
ax[1].set_ylabel("probability")
ax[1].set_title(f"Occupancy distribution at {n_pv} particles per voxel", fontsize=10)
ax[1].legend(frameon=False, fontsize=8.5)

plt.tight_layout(); plt.show()

## What to take from this

The ideal lattice gas reproduces Poisson statistics. Turning on volume exclusion
narrows the distribution by more than a factor of four at $\eta = 0.2$, and the
narrowing matches the compressibility relation with no fitted parameter.

This is a stronger statement than it looks. The first four demonstrations could all
pass with a free energy that had the right *derivative* at the mean density but the
wrong curvature. The variance depends on $d\mu_{\rm ex}/d\eta$, so it probes the
curvature directly. Passing both sets means the functional is right, not just
tangent to right.

It also explains a detail that appears elsewhere in the package. The convergence
metric in `bench/relaxation.py` had to abandon per-voxel profile deviations because
of their Poisson noise floor. Exclusion reduces that floor, but does not remove it,
and the reduction is exactly what is measured here.

**Try changing:**

- `SIGMA_NM = 4` at the same occupancy: a smaller sphere gives a smaller $\eta$, so
  the distribution moves back toward Poisson.
- `OCCUPANCIES = (10,)`: $\eta = 0.34$, where the predicted ratio falls below 0.1.
  Watch the timestep that `suggest_tau` returns.
- `SAMPLE_EVERY = 1`: samples become strongly correlated. The mean and variance are
  unchanged, but they now come from far fewer independent configurations than the
  sample count suggests.
- Set `exclusion=False` while leaving `SIGMA_NM = 8`: the diameters are then ignored
  and the result returns to Poisson. That confirms the effect comes from the free
  energy and not from the occupancy cap.